In [132]:
! pip install contextily
! pip install osmnx

B.) Suppose you and your friend live in different cities on a map and you both are planning for a common meetup place which is optimal for you both. Let’s find the common meetup with a little twist.
On every turn, you and your friend can simultaneously move to a neighboring city on the map. The amount of time needed to move from city i to neighbor j is equal to the straight line distance d(i, j)x2 between the cities, but on each turn the friend that arrives first must wait until the other one arrives (and calls the first on his/her cell phone) before the next turn can begin. The heuristic you are assuming is the straight line distance d(i,j). You both friends want to meet as quickly as possible.


The **MeetingPoint** class processes geographical data to analyze road networks and city locations. It builds a road graph, computes distances between cities, and implements search algorithms like A* and Greedy Best-First Search to find optimal meeting points. Additionally, it provides visualization tools to map cities, routes, and meeting locations efficiently.

In [150]:
import folium
import geopandas as gpd
from geopy.distance import geodesic
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyproj import Transformer
from shapely.geometry import box, Point, LineString
import matplotlib.pyplot as plt
import contextily as ctx
import heapq
import math
import time
import networkx as nx
import osmnx as ox
import numpy as np

class MeetingPoint:
    # Initialize the MeetingPoint class by loading geographical data.
    def __init__(self):
        self.geo_dataframe = gpd.read_file('/content/sample_data/IND_adm2.shp').to_crs(epsg=4326)
        self.road_network = gpd.read_file('/content/sample_data/IND_roads.shp')
        crs_epsg = 32644
        if self.road_network.crs != f'epsg:{crs_epsg}':
            self.road_network = self.road_network.to_crs(epsg=crs_epsg)

    # Builds a road graph from the road network.
    def prepare_road_graph(self):
        road_graph = nx.Graph()
        for _, segment in self.road_network.iterrows():
            if isinstance(segment.geometry, LineString):
                start_node = segment.geometry.coords[0]
                end_node = segment.geometry.coords[-1]
                edge_weight = segment.geometry.length
                road_graph.add_edge(start_node, end_node, weight=edge_weight)
        return road_graph

    # Computes the shortest road distance between two cities.
    @staticmethod
    def get_road_distance(city_a, city_b):
        loc_a = ox.geocode(city_a)
        loc_b = ox.geocode(city_b)
        road_graph = ox.graph_from_point(loc_a, dist=50000, network_type='drive')
        node_a = ox.distance.nearest_nodes(road_graph, loc_a[1], loc_a[0])
        node_b = ox.distance.nearest_nodes(road_graph, loc_b[1], loc_b[0])
        path_length = nx.shortest_path_length(road_graph, node_a, node_b, weight='length')
        return path_length / 1000  # Convert meters to km

    # Finds the coordinates of a city within the road network.
    def get_city_coordinates(self, city_name, city_column='city'):
        city_data = self.road_network[self.road_network[city_column].str.contains(city_name, case=False, na=False)]
        if not city_data.empty:
            city_geometry = city_data.iloc[0].geometry
            return (city_geometry.x, city_geometry.y) if isinstance(city_geometry, Point) else (city_name, city_geometry.coords[0])
        return city_name, None

    # Retrieves geographical data for a specified city.
    def get_geo_data(self, city):
        city_data = self.geo_dataframe[self.geo_dataframe['NAME_2'].str.lower() == city.lower()]
        return city_data

    # Calculates the geodesic distance between two cities.
    def get_distance(self, city_a, city_b):
        city_a_data = self.get_geo_data(city_a)
        city_b_data = self.get_geo_data(city_b)
        if city_a_data.empty or city_b_data.empty:
            return
        geom_a = city_a_data.centroid.iloc[0]
        geom_b = city_b_data.centroid.iloc[0]
        return self.get_distance_between_points(geom_a.y, geom_a.x, geom_b.y, geom_b.x)

    # Finds road connections between two cities.
    def find_edges(self, city_a, city_b):
        print(f"City 1: {city_a} & City 2: {city_b}")
        return self.get_edges(city_a, self.get_neighbouring_city(city_a)), self.get_edges(city_b, self.get_neighbouring_city(city_b))

    # Retrieves the edges connecting a city to its neighboring cities.
    def get_edges(self, city, city_neighbors):
        return [(city, neighbor[0], neighbor[1], neighbor[2]) for neighbor in city_neighbors]

    # Computes the geodesic distance between two geographical points.
    def get_distance_between_points(self, lat1, lon1, lat2, lon2):
        return geodesic((lat1, lon1), (lat2, lon2)).kilometers

    # Estimates the heuristic cost between two cities using Euclidean distance.
    def heuristic(self, city_a, city_b):
        return self.euclidean_distance(city_cords[city_a], city_cords[city_b])

    # Computes the Euclidean distance between two coordinate points.
    def euclidean_distance(self, coord1, coord2):
        lat1, lon1 = coord1
        lat2, lon2 = coord2
        return math.sqrt((lat1 - lat2) ** 2 + (lon1 - lon2) ** 2)

    # Implements the A* search algorithm to find the shortest path between two cities.
    def a_star_search(self, city_a, city_b):
        start = (city_a, city_b)
        frontier = []
        heapq.heappush(frontier, (self.heuristic(city_a, city_b), start, 0, [], [], []))
        visited = set()
        while frontier:
            _, (your_city, friend_city), cost, path, my_trv, friend_trv = heapq.heappop(frontier)
            if (your_city, friend_city) in visited:
                continue
            visited.add((your_city, friend_city))
            if your_city == friend_city:
                return your_city, cost, path + [your_city], my_trv + [your_city], friend_trv + [your_city]
            for you_next in G.neighbors(your_city):
                for friend_next in G.neighbors(friend_city):
                    you_cost = G[your_city][you_next]['weight']
                    friend_cost = G[friend_city][friend_next]['weight']
                    total_cost = cost + max(2 * you_cost, 2 * friend_cost)
                    f_next = total_cost + self.heuristic(you_next, friend_next)
                    heapq.heappush(frontier, (f_next, (you_next, friend_next), total_cost, path + [your_city], my_trv + [your_city], friend_trv + [friend_city]))
        return None

**Q1. Formulate this search problem and display the map of the city with heuristic (h) and transition cost (g).**

In [135]:
main = MeetingPoint()
my_start_city = 'Naini Tal'
friend_start_city='Indore'
city1_edges, city2_edges = main.find_edges(my_start_city, friend_start_city)

City 1: Naini Tal & City 2: Indore
City 1: 29.317087269390218, 79.45483872140413
City 2: 22.71663343073364, 75.78138525626387
City 1: Naini Tal & City 2: Indore are 818.35 km apart.


<ipython-input-133-17b1336bde93>:163: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  neighboring_cities = self.gdf[self.gdf.geometry.centroid.within(range_buffer_gdx.geometry.iloc[0])]


Neighboring cities within the range is : 68
Neighboring cities within the range is : 195
Neighboring cities within the range is : 62


<ipython-input-133-17b1336bde93>:163: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  neighboring_cities = self.gdf[self.gdf.geometry.centroid.within(range_buffer_gdx.geometry.iloc[0])]


Neighboring cities within the range is : 68
Neighboring cities within the range is : 193
Neighboring cities within the range is : 56
City : Naini Tal --240.38966855854113-->Delhi
City : Naini Tal --346.96987121383-->Bhiwani
City : Naini Tal --244.30593418221898-->Faridabad
City : Naini Tal --272.89645124140856-->Gurgaon
City : Naini Tal --354.8842589523224-->Hisar
City : Naini Tal --288.0955608621943-->Jhajjar
City : Naini Tal --347.82221093206397-->Mahendragarh
City : Naini Tal --246.65683204314234-->Panipat
City : Naini Tal --308.5769012218975-->Rewari
City : Naini Tal --289.96812750245306-->Rohtak
City : Naini Tal --253.50346502037272-->Sonepat
City : Naini Tal --544.3364462977172-->Ashoknagar
City : Naini Tal --328.5088994929357-->Bhind
City : Naini Tal --678.4343732179804-->Bhopal
City : Naini Tal --609.5099287808855-->Damoh
City : Naini Tal --393.71825638403925-->Datia
City : Naini Tal --572.8724955117243-->Guna
City : Naini Tal --385.12079854794854-->Gwalior
City : Naini Tal --3

In [136]:
def calculate_all_edges_within_region(byroad):
  if byroad:
      all_edges_within_region=main.get_all_edges()
  else:
      all_edges_within_region = city1_edges + city2_edges
  print(city1_edges)
  print(city2_edges)
  print(all_edges_within_region)
  city_distances = []
  # Graph connections for both
  G = nx.Graph()
  all_edges = []

  if byroad:
      road_g=main.prepare_road_graph()
      G = road_g
      for u, v, w, loc in all_edges_within_region:
          if u != v:
              is_road_connected = main.is_road_connected(main.get_geo_data(u), main.get_geo_data(v),
                                                         main.roads_gdf)
              if is_road_connected:
                  all_edges.append((u,v,w,loc))
      all_edges_within_region=all_edges

  for u, v, w, loc in all_edges_within_region:
      if u != v:
          G.add_edge(u, v, weight=w)
          G.add_node(u)
          city_distances.append((v,loc))
          city_distances.append((u, loc))

  for u, v, w, loc in city1_edges + city2_edges:
      city_distances.append((v,loc))
  global city_cords
  city_cords = dict(city_distances)
  print(city_cords)

  city_cords_roads_list = []
  for city in city_cords.items():
      city_cords_roads_list.append((city[0],main.get_city_coordinates(city[0])))

  city_cords_roads = dict(city_cords_roads_list)

  main.plot_graph()

In [137]:
def show_result(result, name, is_road_enable):
  if result:
      meet, cost, nodes, space, duration, path, my_trv_path, friend_trv_path = result
      print(f" My City City: {my_start_city} & Friend City: {friend_start_city}")
      print(f" Meeting City: {meet}")
      print(f"Total Cost: {cost}")
      print(f"Nodes Generated: {nodes}")
      print(f"Max Frontier Size: {space}")
      print(f"Execution Time: {duration:.4f} sec")
      print(f"Path: {path}")
      print(f"My Path: {my_trv_path}")
      print(f"Friend Path: {friend_trv_path}")
      main.plot_graph_goal(G,meet)
      main.plot_cities(city1_edges, city2_edges, meet, None)
      # Plot the map

      m = main.plot_result_on_map(my_trv_path, friend_trv_path, meet, city_cords)
      byroad = "straight_line"
      if is_road_enable:
        byroad = "by_road"
      m.save(f"{name}_{byroad}.html")
      # Save and display
      display.display(m)

In [138]:
def bfs_result(is_road_enable):
  print()
  print(f"GBFS Result with Road enable = {is_road_enable}")
  result = main.greedy_best_first_search(my_start_city,friend_start_city)
  show_result(result, "gbfs", is_road_enable)

def a_start_result(is_road_enable):
  print()
  print(f"A* Result with Road enable = {is_road_enable}")
  result = main.a_star_search(my_start_city,friend_start_city)
  if result:
      show_result(result, "a_start", is_road_enable)

**Q2**. **Implement search strategies:** a. Greedy Best First Search b. A* and provide the Search Cost nodes generated, space and time taken for execution  for both of them.

Before this I have created method that calulcate heuristic value. Aslo based on that values bfs and A* will perform. Here we manger the heuristic by varible **is_road_enable**. When **is_road_enable = false it will do search based on straight Line**

In [139]:
is_road_enable = False
calculate_all_edges_within_region(is_road_enable)

[('Naini Tal', 'Delhi', 240.38966855854113, (28.646518974169425, 77.1089795345416)), ('Naini Tal', 'Bhiwani', 346.96987121383, (28.72409633312063, 75.95756044709584)), ('Naini Tal', 'Faridabad', 244.30593418221898, (28.16405334743672, 77.3231410391257)), ('Naini Tal', 'Gurgaon', 272.89645124140856, (28.168540944445407, 76.983476257174)), ('Naini Tal', 'Hisar', 354.8842589523224, (29.220394929158005, 75.8047498264771)), ('Naini Tal', 'Jhajjar', 288.0955608621943, (28.5923754917194, 76.61653315999435)), ('Naini Tal', 'Mahendragarh', 347.82221093206397, (28.17826088333133, 76.13633286445628)), ('Naini Tal', 'Panipat', 246.65683204314234, (29.33742151880353, 76.91545499008622)), ('Naini Tal', 'Rewari', 308.5769012218975, (28.210452306358892, 76.55527760624655)), ('Naini Tal', 'Rohtak', 289.96812750245306, (28.914147348459654, 76.51108115795722)), ('Naini Tal', 'Sonepat', 253.50346502037272, (29.06547089672502, 76.86412452228555)), ('Naini Tal', 'Ashoknagar', 544.3364462977172, (24.61411818

In [140]:
bfs_result(is_road_enable)


GBFS Result with Road enable = False
 My City City: Naini Tal & Friend City: Indore
 Meeting City: Bhind
Total Cost: 1272.2992860542645
Nodes Generated: 177
Max Frontier Size: 170
Execution Time: 0.0011 sec
Path: ['Naini Tal-Indore', 'Rampur-Dewas', 'Badaun-Sehore', 'Etah-Bhopal', 'Etawah-Vidisha', 'Bhind-Ashoknagar', 'Gwalior-Shivpuri', 'Morena-Gwalior', 'Bhind']
My Path: ['Naini Tal', 'Rampur', 'Badaun', 'Etah', 'Etawah', 'Bhind', 'Gwalior', 'Morena', 'Bhind']
Friend Path: ['Indore', 'Dewas', 'Sehore', 'Bhopal', 'Vidisha', 'Ashoknagar', 'Shivpuri', 'Gwalior', 'Bhind']
Meeting points: (26.426501123712743, 78.71485018127507)


In [141]:
a_start_result(is_road_enable)


A* Result with Road enable = False
 My City City: Naini Tal & Friend City: Indore
 Meeting City: Gwalior
Total Cost: 1029.454798205413
Nodes Generated: 16935
Max Frontier Size: 8176
Execution Time: 0.4099 sec
Path: ['Naini Tal-Indore', 'Rampur-Dewas', 'Badaun-Sehore', 'Etah-Bhopal', 'Firozabad-Vidisha', 'Agra-Ashoknagar', 'Morena-Shivpuri', 'Gwalior']
My Path: ['Naini Tal', 'Rampur', 'Badaun', 'Etah', 'Firozabad', 'Agra', 'Morena', 'Gwalior']
Friend Path: ['Indore', 'Dewas', 'Sehore', 'Bhopal', 'Vidisha', 'Ashoknagar', 'Shivpuri', 'Gwalior']
Meeting points: (26.045045627381565, 78.13879230963495)


Now chnaging the value of **is_road_enable** to True, it will do search based on road available b/w the cities

In [147]:
is_road_enable = True
calculate_all_edges_within_region(is_road_enable)

[('Naini Tal', 'Delhi', 240.38966855854113, (28.646518974169425, 77.1089795345416)), ('Naini Tal', 'Bhiwani', 346.96987121383, (28.72409633312063, 75.95756044709584)), ('Naini Tal', 'Faridabad', 244.30593418221898, (28.16405334743672, 77.3231410391257)), ('Naini Tal', 'Gurgaon', 272.89645124140856, (28.168540944445407, 76.983476257174)), ('Naini Tal', 'Hisar', 354.8842589523224, (29.220394929158005, 75.8047498264771)), ('Naini Tal', 'Jhajjar', 288.0955608621943, (28.5923754917194, 76.61653315999435)), ('Naini Tal', 'Mahendragarh', 347.82221093206397, (28.17826088333133, 76.13633286445628)), ('Naini Tal', 'Panipat', 246.65683204314234, (29.33742151880353, 76.91545499008622)), ('Naini Tal', 'Rewari', 308.5769012218975, (28.210452306358892, 76.55527760624655)), ('Naini Tal', 'Rohtak', 289.96812750245306, (28.914147348459654, 76.51108115795722)), ('Naini Tal', 'Sonepat', 253.50346502037272, (29.06547089672502, 76.86412452228555)), ('Naini Tal', 'Ashoknagar', 544.3364462977172, (24.61411818

In [148]:
bfs_result(is_road_enable)


GBFS Result with Road enable = True
 My City City: Naini Tal & Friend City: Indore
 Meeting City: Bhind
Total Cost: 1272.2992860542645
Nodes Generated: 177
Max Frontier Size: 170
Execution Time: 0.0007 sec
Path: ['Naini Tal-Indore', 'Rampur-Dewas', 'Badaun-Sehore', 'Etah-Bhopal', 'Etawah-Vidisha', 'Bhind-Ashoknagar', 'Gwalior-Shivpuri', 'Morena-Gwalior', 'Bhind']
My Path: ['Naini Tal', 'Rampur', 'Badaun', 'Etah', 'Etawah', 'Bhind', 'Gwalior', 'Morena', 'Bhind']
Friend Path: ['Indore', 'Dewas', 'Sehore', 'Bhopal', 'Vidisha', 'Ashoknagar', 'Shivpuri', 'Gwalior', 'Bhind']
Meeting points: (26.426501123712743, 78.71485018127507)


In [149]:
a_start_result(is_road_enable)


A* Result with Road enable = True
 My City City: Naini Tal & Friend City: Indore
 Meeting City: Gwalior
Total Cost: 1029.454798205413
Nodes Generated: 16935
Max Frontier Size: 8176
Execution Time: 0.0966 sec
Path: ['Naini Tal-Indore', 'Rampur-Dewas', 'Badaun-Sehore', 'Etah-Bhopal', 'Firozabad-Vidisha', 'Agra-Ashoknagar', 'Morena-Shivpuri', 'Gwalior']
My Path: ['Naini Tal', 'Rampur', 'Badaun', 'Etah', 'Firozabad', 'Agra', 'Morena', 'Gwalior']
Friend Path: ['Indore', 'Dewas', 'Sehore', 'Bhopal', 'Vidisha', 'Ashoknagar', 'Shivpuri', 'Gwalior']
Meeting points: (26.045045627381565, 78.13879230963495)
